In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")

from src.recommender.knn import ItemKNN

engine = create_engine(
    f"postgresql+psycopg://{os.environ['POSTGRES_USER']}:{os.environ['POSTGRES_PASSWORD']}"
    f"@{os.getenv('POSTGRES_HOST', 'localhost')}:{os.getenv('POSTGRES_PORT', '5432')}/{os.environ['POSTGRES_DB']}"
)

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 200)

products = pd.read_sql("SELECT product_id, product_name, aisle, department FROM raw.v_products", engine).set_index("product_id")
print("Products:", len(products))
print("Embeddings in DB:", pd.read_sql("SELECT count(*) AS n FROM app.product_embeddings", engine)["n"][0])

In [ ]:
train = pd.read_csv(ROOT / "data/processed/train.csv")
knn = ItemKNN().fit(train, history_users=[])
popularity = train.groupby("product_id")["n_orders"].sum()

In [ ]:
def find(query, limit=15):
    hits = products[products["product_name"].str.contains(query, case=False, regex=False)].copy()
    hits["popularity"] = popularity.reindex(hits.index).fillna(0).astype(int)
    return hits.sort_values("popularity", ascending=False).head(limit)


def ids_by_name(names):
    ids = []
    for name in names:
        match = products[products["product_name"] == name]
        if match.empty:
            raise ValueError(f"Product not found: {name!r}. Use find('...') to search.")
        ids.append(int(match.index[0]))
    return ids

In [ ]:
SIMILAR_SQL = text("""
SELECT e.product_id, p.product_name, p.aisle, p.department,
       1 - (e.embedding <=> q.v) AS similarity
FROM app.product_embeddings e
JOIN raw.v_products p USING (product_id),
     (SELECT avg(embedding) AS v FROM app.product_embeddings WHERE product_id = ANY(:ids)) q
WHERE e.product_id <> ALL(:ids)
ORDER BY e.embedding <=> q.v
LIMIT :k
""")


def similar(product_ids, k=10):
    if isinstance(product_ids, int):
        product_ids = [product_ids]
    return pd.read_sql(SIMILAR_SQL, engine, params={"ids": [int(p) for p in product_ids], "k": k}).set_index("product_id")

In [ ]:
def copurchase(product_ids, k=10):
    scores = {}
    cart = {int(p) for p in product_ids}
    for p in cart:
        idx = knn.item_to_idx.get(p)
        if idx is None:
            continue
        for neighbor, sim in zip(knn.neighbor_idx[idx], knn.neighbor_sim[idx]):
            item = int(knn.items[neighbor])
            if item not in cart and sim > 0:
                scores[item] = scores.get(item, 0.0) + float(sim)
    ranked = sorted(scores, key=lambda i: -scores[i])[:k]
    return pd.Series({i: scores[i] for i in ranked}, name="score")


def recommend_cart(names, k=10):
    ids = ids_by_name(names)
    text_recs = similar(ids, k=50)
    co_recs = copurchase(ids, k=50)

    fused = {}
    for rank, item in enumerate(text_recs.index):
        fused[item] = fused.get(item, 0.0) + 1 / (60 + rank)
    for rank, item in enumerate(co_recs.index):
        fused[item] = fused.get(item, 0.0) + 1 / (60 + rank)
    hybrid = sorted(fused, key=lambda i: -fused[i])[:k]

    def label(items):
        return [f"{products.loc[i, 'product_name']} [{products.loc[i, 'department']}]" for i in items][:k]

    print("CART:", ", ".join(names), "\n")
    return pd.DataFrame({
        "text (embeddings)": label(text_recs.index),
        "co-purchase (KNN)": label(co_recs.index) + [""] * (k - len(co_recs.index[:k])),
        "hybrid": label(hybrid),
    })

In [ ]:
find("milk")

In [ ]:
similar(ids_by_name(["Bag of Organic Bananas"]))

In [ ]:
CART_NAMES = ["Bag of Organic Bananas", "Organic Whole Milk", "Organic Hass Avocado"]

recommend_cart(CART_NAMES, k=10)